In [ ]:
from __future__ import annotations

import re
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable

# ── Font / style (matches experiment_1_set_consistency/plots.ipynb) ──────────
LABEL_FONTSIZE           = 28
TITLE_FONTSIZE           = 32
TICK_FONTSIZE            = 26
CATEGORY_TICK_FONTSIZE   = 22
ROWLABEL_FONTSIZE        = 22
SUPTITLE_FONTSIZE        = 32
CBAR_TICK_FONTSIZE       = 24
CBAR_LABEL_FONTSIZE      = 28
LEGEND_FONTSIZE          = 20

HEATMAP_CMAP = 'RdBu_r'
VMIN, VMAX   = -1.0, 1.0

# Subplot layout (matches original)
MAIN_SUBPLOT_LEFT   = 0.10
MAIN_SUBPLOT_RIGHT  = 0.945
MAIN_SUBPLOT_BOTTOM = 0.045
MAIN_SUBPLOT_TOP    = 0.89

MAX_CATEGORIES = 60

# Main grid: 5 data panels per row (4 rectangular + 1 square)
# Columns: [typ-ctx | typ-Jac | emb-Jac | ctx-Jac | SPACER | layer-layer]
COLUMN_TITLES = [
    r'$\rho_{\mathrm{S}}(s_{\mathrm{hum}},\, s_{\mathrm{cos}}^\ell)$',
    r'$\rho_{\mathrm{S}}(s_{\mathrm{hum}},\, s_{\mathrm{Jac}}^\ell)$',
    r'$\rho_{\mathrm{S}}(s_{\mathrm{emb}},\, s_{\mathrm{Jac}}^\ell)$',
    r'$\rho_{\mathrm{S}}(s_{\mathrm{cos}}^\ell,\, s_{\mathrm{Jac}}^\ell)$',
    r'$\rho_{\mathrm{S}}(s_{\mathrm{Jac}}^{\ell_1},\, s_{\mathrm{Jac}}^{\ell_2})$',
]

MODEL_ORDER = [
    'gemma-3-270m', 'gemma-3-1b-pt', 'gemma-3-4b-pt',
    'gemma-3-12b-pt', 'gemma-3-27b-pt',
    'gpt2-small', 'meta-llama/Llama-3.1-8B', 'mistralai/Mistral-7B-v0.1',
]

PRETTY_MODEL = {
    'gemma-3-270m':                'Gemma 3 270M',
    'gemma-3-1b-pt':               'Gemma 3 1B',
    'gemma-3-4b-pt':               'Gemma 3 4B',
    'gemma-3-12b-pt':              'Gemma 3 12B',
    'gemma-3-27b-pt':              'Gemma 3 27B',
    'gpt2-small':                  'GPT-2 Small',
    'meta-llama/Llama-3.1-8B':     'Llama 3.1 8B',
    'mistralai/Mistral-7B-v0.1':   'Mistral 7B',
}

# ── Special-case configuration ───────────────────────────────────────────────
# Rows in this set show reference text instead of heatmaps in the main grid.
EXCLUDE_FROM_MAIN = {("gemma-3-270m", "16k big")}
SPECIAL_CASES = [("gemma-3-270m", "16k big")]
SPECIAL_REFERENCE_TEXT = {
    ("gemma-3-270m", "16k big"): (
        "see Fig. 2\n(panel b)",   # panel 1: typicality vs Jaccard
        "see Fig. 2\n(panel c)",   # panel 2: ctx cosim vs Jaccard
    ),
}

# ── Paths ────────────────────────────────────────────────────────────────────
RESULTS_DIR = Path('../../results/experiment_1_typicality')  # base_save_dir of main.py, relative to this notebook
OUTPUT_DIR  = Path('paper_plots_out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Discovery + loading ──────────────────────────────────────────────────────

def find_all_result_dirs(base: Path) -> List[Path]:
    """All directories under base that contain paper-metrics CSV output."""
    return [p.parent for p in sorted(base.rglob('df_cat_typ_emb.csv'))]


def parse_run_meta(result_dir: Path, base: Path) -> Tuple[str, str]:
    """Return (model_name, sae_release) from a result directory path."""
    rel   = result_dir.relative_to(base)
    parts = rel.parts
    # Handle org/model paths (e.g. mistralai/Mistral-7B-v0.1 -> two parts)
    if len(parts) >= 3 and not any(c.isdigit() for c in parts[0]) and '/' not in parts[0]:
        candidate = f"{parts[0]}/{parts[1]}"
        if candidate in MODEL_ORDER:
            return candidate, parts[2] if len(parts) > 2 else 'unknown'
    model_name  = parts[0] if len(parts) > 0 else 'unknown'
    sae_release = parts[1] if len(parts) > 1 else 'unknown'
    return model_name, sae_release


def _read_sae_label(result_dir: Path) -> Optional[str]:
    """Extract SAE display label from summary.txt 'SAE id template:' line."""
    p = result_dir / 'summary.txt'
    if not p.exists():
        return None
    for line in p.read_text(encoding='utf-8').splitlines():
        if line.startswith('SAE id template:'):
            id_template = line.split(':', 1)[1].strip()
            s = id_template.lower()
            m = re.search(r'(\d+k).+?(big|small|medium|canonical)', s)
            if m:
                return f"{m.group(1)} {m.group(2)}"
            m = re.search(r'(\d+k)', s)
            if m:
                return m.group(1)
    return None


def sae_row_label(sae_release: str) -> str:
    s = sae_release.lower()
    m = re.search(r'(\d+k)[_-](big|small|medium|canonical)', s)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    m = re.search(r'(\d+k)', s)
    if m:
        return m.group(1)
    parts = [p for p in sae_release.replace('_', '-').split('-') if p]
    return '-'.join(parts[-2:]) if len(parts) >= 2 else sae_release


def sae_sort_key(label: str):
    m = re.match(r'(\d+)k(?:\s+(big|small|medium|canonical))?$', label)
    if m:
        size = int(m.group(1))
        var  = {'big': 0, 'medium': 1, 'small': 2, 'canonical': 3}.get(m.group(2), 3)
        return (size, var, label)
    return (999, 999, label)


def model_sort_key(name: str) -> int:
    return MODEL_ORDER.index(name) if name in MODEL_ORDER else 10**9


def load_tables(result_dir: Path) -> dict:
    def _read(name):
        p = result_dir / name
        return pd.read_csv(p) if p.exists() else None
    return {
        'df_cat_typ_emb':          _read('df_cat_typ_emb.csv'),
        'df_cat_typ_ctx_by_layer': _read('df_cat_typ_ctx_by_layer.csv'),
        'df_cat_typ_jac_by_layer': _read('df_cat_typ_jac_by_layer.csv'),
        'df_cat_emb_jac_by_layer': _read('df_cat_emb_jac_by_layer.csv'),
        'df_cat_ctx_jac_by_layer': _read('df_cat_ctx_jac_by_layer.csv'),
        'df_ll_jac':               _read('df_ll_jac.csv'),
    }


def load_tables_apt(result_dir: Path) -> dict:
    """Load APM-based metrics from apt_sym result directories.

    Tries both the old naming convention (per_category_spearman_*.csv,
    layer_layer_spearman_apm_scores.csv) and the new naming convention
    (df_cat_typ_apm_by_layer.csv, df_cat_cos_apm_by_layer.csv, df_ll.csv).
    """
    def _read(name):
        p = result_dir / name
        return pd.read_csv(p) if p.exists() else None

    # Layer-layer matrix: old name first, then new name
    ll = _read('layer_layer_spearman_apm_scores.csv')
    if ll is not None and 'Unnamed: 0' in ll.columns:
        ll = ll.set_index('Unnamed: 0')
        ll.index = pd.Index([str(int(i)) for i in ll.index])
    if ll is None:
        ll = _read('df_ll.csv')

    # Per-category by-layer: old names first, fall back to new names
    _a = _read('per_category_spearman_typ_vs_apm_by_layer.csv')
    typ_jac = _a if _a is not None else _read('df_cat_typ_apm_by_layer.csv')
    _b = _read('per_category_spearman_cosim_vs_apm_by_layer.csv')
    cos_jac = _b if _b is not None else _read('df_cat_cos_apm_by_layer.csv')

    # Static embedding baseline (aggregate spearman vs emb cosim)
    typ_emb = _read('df_cat_typ_cos.csv')

    return {
        'df_cat_typ_emb':          typ_emb,
        'df_cat_typ_ctx_by_layer': None,   # ctx cosim not computed in apt_sym runs
        'df_cat_typ_jac_by_layer': typ_jac,
        'df_cat_emb_jac_by_layer': cos_jac,
        'df_cat_ctx_jac_by_layer': None,   # ctx-jac not computed in apt_sym runs
        'df_ll_jac':               ll,
    }


def draw_reference_panel(ax: Axes, text: str) -> None:
    """Draw a placeholder panel with reference text (e.g. 'see Fig. 1')."""
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True)
        sp.set_alpha(0.25)
    ax.set_facecolor('#f8f8f8')
    ax.text(0.5, 0.5, text,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=LABEL_FONTSIZE, style='italic', color='#555555',
            multialignment='center')

def get_layer_cols(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r'layer_\d+', str(c))]
    return sorted(cols, key=lambda c: int(c.split('_')[1]))


def normalize_ll(df: pd.DataFrame) -> pd.DataFrame:
    if 'index' in df.columns:
        df = df.set_index('index')
    # Ensure both index and columns are strings so intersection works
    df.index   = pd.Index([str(c) for c in df.index])
    df.columns = [str(c) for c in df.columns]
    common = sorted([c for c in df.columns if c in set(df.index)],
                    key=lambda x: int(x) if x.isdigit() else x)
    if common:
        df = df.loc[common, common]
    return df.astype(float)


def sparse_xticks(vals: List[int], n: int = 6) -> List[int]:
    vals = sorted(vals)
    if len(vals) <= n:
        return vals
    idxs = [round(i * (len(vals) - 1) / (n - 1)) for i in range(n)]
    seen, out = set(), []
    for idx in idxs:
        v = vals[idx]
        if v not in seen:
            out.append(v); seen.add(v)
    return out


# ── Drawing primitives ────────────────────────────────────────────

def _cmap_with_nan():
    cmap = plt.get_cmap(HEATMAP_CMAP).copy()
    cmap.set_bad(color='white')
    return cmap


def draw_blank(ax: Axes) -> None:
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_alpha(0.25)


def plot_rect_on_ax(
    ax: Axes, mat: np.ndarray,
    row_labels: List[str], col_labels: List[str],
    title: str = '', xlabel: str = '',
    show_title: bool = False, show_xlabel: bool = False,
    show_yticks: bool = True, max_xticks: int = 6,
) -> None:
    norm = mcolors.TwoSlopeNorm(vmin=VMIN, vcenter=0.0, vmax=VMAX)
    ax.imshow(mat, aspect='auto', cmap=_cmap_with_nan(), norm=norm, interpolation='nearest')

    col_vals = [int(x) for x in col_labels]
    ticks    = sparse_xticks(col_vals, n=max_xticks)
    tidx     = [col_vals.index(x) for x in ticks]
    ax.set_xticks(tidx)
    ax.set_xticklabels([str(x) for x in ticks] if show_xlabel else ['']*len(tidx),
                       fontsize=TICK_FONTSIZE)

    if show_yticks:
        ax.set_yticks(range(len(row_labels)))
        ax.set_yticklabels(row_labels, fontsize=CATEGORY_TICK_FONTSIZE)
    else:
        ax.set_yticks([])

    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE, pad=4)
    ax.set_xlabel(xlabel if show_xlabel else '', fontsize=LABEL_FONTSIZE)
    ax.set_xlim(-0.5, mat.shape[1] - 0.5)
    ax.set_ylim(mat.shape[0] - 0.5, -0.5)
    if show_title:
        ax.set_title(title, fontsize=TITLE_FONTSIZE, pad=18)


def plot_ll_on_ax(
    ax: Axes, df_ll: pd.DataFrame,
    title: str = '',
    show_title: bool = False, show_xlabel: bool = False,
    max_ticks: int = 6,
) -> None:
    mat    = df_ll.to_numpy(dtype=float)
    labels = [str(c) for c in df_ll.columns]
    mat_p  = mat.copy()
    mat_p[np.triu(np.ones_like(mat, dtype=bool), k=1)] = np.nan

    norm = mcolors.TwoSlopeNorm(vmin=VMIN, vcenter=0.0, vmax=VMAX)
    ax.imshow(mat_p, aspect='equal', cmap=_cmap_with_nan(), norm=norm, interpolation='nearest')

    tick_vals = [int(x) for x in labels]
    ticks     = sparse_xticks(tick_vals, n=max_ticks)
    tidx      = [tick_vals.index(x) for x in ticks]
    tlabels   = [str(x) for x in ticks]

    ax.set_xticks(tidx)
    ax.set_xticklabels(tlabels if show_xlabel else ['']*len(tidx), fontsize=TICK_FONTSIZE)
    ax.set_yticks(tidx)
    ax.set_yticklabels(tlabels, fontsize=TICK_FONTSIZE)
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE, pad=4)
    ax.set_xlabel(r'Layer $\ell_1$' if show_xlabel else '', fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(r'Layer $\ell_2$', fontsize=LABEL_FONTSIZE)
    ax.set_xlim(-0.5, mat.shape[1] - 0.5)
    ax.set_ylim(mat.shape[0] - 0.5, -0.5)
    if show_title:
        ax.set_title(title, fontsize=TITLE_FONTSIZE, pad=18)


def add_shared_colorbar(
    fig, bottom: float, top: float,
    x: float = 0.955, width: float = 0.012,
) -> None:
    norm = mcolors.TwoSlopeNorm(vmin=VMIN, vcenter=0.0, vmax=VMAX)
    sm   = ScalarMappable(norm=norm, cmap=plt.get_cmap(HEATMAP_CMAP))
    sm.set_array([])
    cax  = fig.add_axes([x, bottom, width, top - bottom])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label(r'Spearman $\rho$', fontsize=CBAR_LABEL_FONTSIZE, labelpad=12)
    cbar.ax.tick_params(labelsize=CBAR_TICK_FONTSIZE)


# ── Row rendering ─────────────────────────────────────────────────────
# Subplot column indices: 0=ctx, 1=jac, 2=emb_jac, 3=ctx_jac, 4=SPACER, 5=layer-layer

def plot_row(
    axes,
    tables: dict,
    row_label: str,
    ordered_cats: List[str],
    show_titles: bool,
    last_row: bool,
    reference_text_panel1: Optional[str] = None,
    reference_text_panel2: Optional[str] = None,
) -> None:
    ax_ctx, ax_jac, ax_ej, ax_cj, ax_sp, ax_ll = axes
    ax_sp.axis('off')

    ax_ctx.text(
        -0.75, 0.5, row_label,
        transform=ax_ctx.transAxes, rotation=90,
        ha='center', va='center',
        fontsize=ROWLABEL_FONTSIZE, fontweight='bold', clip_on=False,
    )

    # Derive layer_cols from the first available by-layer table.
    # apt_sym rows have no df_cat_typ_ctx_by_layer, so we must check all.
    _lcol_candidates = [
        tables.get('df_cat_typ_ctx_by_layer'),
        tables.get('df_cat_typ_jac_by_layer'),
        tables.get('df_cat_emb_jac_by_layer'),
        tables.get('df_cat_ctx_jac_by_layer'),
    ]
    layer_cols   = next((get_layer_cols(df) for df in _lcol_candidates if df is not None), [])
    layer_labels = [str(int(c.split('_')[1])) for c in layer_cols]

    # ── Panel 0: typicality vs ctx cosim by layer ─────────────────────────────
    if tables['df_cat_typ_ctx_by_layer'] is not None and layer_cols:
        _lc = get_layer_cols(tables['df_cat_typ_ctx_by_layer'])
        _ll = [str(int(c.split('_')[1])) for c in _lc]
        df  = tables['df_cat_typ_ctx_by_layer'].set_index('category').reindex(ordered_cats)
        mat = df[_lc].to_numpy(dtype=float)
        plot_rect_on_ax(
            ax_ctx, mat, ordered_cats, _ll,
            title=COLUMN_TITLES[0], xlabel=r'Layer $\ell$',
            show_title=show_titles, show_xlabel=last_row, show_yticks=True,
        )
    else:
        draw_blank(ax_ctx)
        if show_titles: ax_ctx.set_title(COLUMN_TITLES[0], fontsize=TITLE_FONTSIZE, pad=18)

    # ── Panel 1: typicality vs Jaccard by layer ────────────────────────────
    if reference_text_panel1 is not None:
        draw_reference_panel(ax_jac, reference_text_panel1)
        if show_titles: ax_jac.set_title(COLUMN_TITLES[1], fontsize=TITLE_FONTSIZE, pad=18)
    elif tables['df_cat_typ_jac_by_layer'] is not None and layer_cols:
        _lc = get_layer_cols(tables['df_cat_typ_jac_by_layer'])
        _ll = [str(int(c.split('_')[1])) for c in _lc]
        df  = tables['df_cat_typ_jac_by_layer'].set_index('category').reindex(ordered_cats)
        mat = df[_lc].to_numpy(dtype=float)
        plot_rect_on_ax(
            ax_jac, mat, ordered_cats, _ll,
            title=COLUMN_TITLES[1], xlabel=r'Layer $\ell$',
            show_title=show_titles, show_xlabel=last_row, show_yticks=False,
        )
    else:
        draw_blank(ax_jac)
        if show_titles: ax_jac.set_title(COLUMN_TITLES[1], fontsize=TITLE_FONTSIZE, pad=18)

    # ── Panel 2: emb cosim vs Jaccard by layer ────────────────────────────
    if tables['df_cat_emb_jac_by_layer'] is not None and layer_cols:
        _lc = get_layer_cols(tables['df_cat_emb_jac_by_layer'])
        _ll = [str(int(c.split('_')[1])) for c in _lc]
        df  = tables['df_cat_emb_jac_by_layer'].set_index('category').reindex(ordered_cats)
        mat = df[_lc].to_numpy(dtype=float)
        plot_rect_on_ax(
            ax_ej, mat, ordered_cats, _ll,
            title=COLUMN_TITLES[2], xlabel=r'Layer $\ell$',
            show_title=show_titles, show_xlabel=last_row, show_yticks=False,
        )
    else:
        draw_blank(ax_ej)
        if show_titles: ax_ej.set_title(COLUMN_TITLES[2], fontsize=TITLE_FONTSIZE, pad=18)

    # ── Panel 3: ctx cosim vs Jaccard at same layer ────────────────────────
    if reference_text_panel2 is not None:
        draw_reference_panel(ax_cj, reference_text_panel2)
        if show_titles: ax_cj.set_title(COLUMN_TITLES[3], fontsize=TITLE_FONTSIZE, pad=18)
    elif tables['df_cat_ctx_jac_by_layer'] is not None and layer_cols:
        _lc = get_layer_cols(tables['df_cat_ctx_jac_by_layer'])
        _ll = [str(int(c.split('_')[1])) for c in _lc]
        df  = tables['df_cat_ctx_jac_by_layer'].set_index('category').reindex(ordered_cats)
        mat = df[_lc].to_numpy(dtype=float)
        plot_rect_on_ax(
            ax_cj, mat, ordered_cats, _ll,
            title=COLUMN_TITLES[3], xlabel=r'Layer $\ell$',
            show_title=show_titles, show_xlabel=last_row, show_yticks=False,
        )
    else:
        draw_blank(ax_cj)
        if show_titles: ax_cj.set_title(COLUMN_TITLES[3], fontsize=TITLE_FONTSIZE, pad=18)

    # ── Panel 5: layer x layer Jaccard ───────────────────────────────────────
    if tables['df_ll_jac'] is not None:
        plot_ll_on_ax(
            ax_ll, normalize_ll(tables['df_ll_jac']),
            title=COLUMN_TITLES[4],
            show_title=show_titles, show_xlabel=last_row,
        )
    else:
        draw_blank(ax_ll)
        if show_titles: ax_ll.set_title(COLUMN_TITLES[4], fontsize=TITLE_FONTSIZE, pad=18)

# Llama Scope 32x has 2**17 = 131,072 latents and is called "128k" by its release, like GPT-2 v5-128k:
# label both "128k". Older runs/configs call the Llama SAE "131k".
def _width_128k(label):
    return "128k" if label == "131k" else label


__read_sae_label_raw = _read_sae_label
_sae_row_label_raw = sae_row_label


def _read_sae_label(*args, **kwargs):
    return _width_128k(__read_sae_label_raw(*args, **kwargs))


def sae_row_label(*args, **kwargs):
    return _width_128k(_sae_row_label_raw(*args, **kwargs))


In [ ]:
# ── Discover runs ─────────────────────────────────────────────────────────────
result_dirs = find_all_result_dirs(RESULTS_DIR)
if not result_dirs:
    raise FileNotFoundError(
        f'No paper-metrics results found under {RESULTS_DIR}.\n'
        'Run main.py with the experiment_2_set_consistency config first.'
    )

print(f'Found {len(result_dirs)} result director{"y" if len(result_dirs)==1 else "ies"}:')
for d in result_dirs:
    print(f'  {d}')

# Group by model
model_to_runs: Dict[str, Dict[str, dict]] = defaultdict(dict)
for rdir in result_dirs:
    model_name, sae_release = parse_run_meta(rdir, RESULTS_DIR)
    row_label = _read_sae_label(rdir) or sae_row_label(sae_release)
    model_to_runs[model_name][row_label] = load_tables(rdir)

# Global category order: alphabetical (matches experiment_1_set_consistency)
rho_acc: Dict[str, List[float]] = defaultdict(list)
for runs in model_to_runs.values():
    for T in runs.values():
        df = T['df_cat_typ_jac_by_layer']
        if df is None: continue
        lcols = get_layer_cols(df)
        for _, row in df.iterrows():
            rho_acc[str(row['category'])].append(
                float(np.nanmean([row[c] for c in lcols]))
            )

ordered_cats = sorted(rho_acc.keys())[:MAX_CATEGORIES]
print(f'\nCategories: {len(ordered_cats)}, first 10: {ordered_cats[:10]}')


In [ ]:
# ── Load additional data from W&B (experiment_2_set_consistency_paper) ────────
import json as _json
import wandb

import os
import wandb as _wandb_for_entity
# W&B entity: $WANDB_ENTITY if set, otherwise your default entity.
WANDB_ENTITY          = os.environ.get("WANDB_ENTITY") or _wandb_for_entity.Api().default_entity
WANDB_PROJECT         = 'experiment_1_typicality'  # wandb.project in configs/experiment_1_typicality/base.yaml
WANDB_CACHE           = Path('.wandb_cache_paper')
WANDB_CACHE.mkdir(parents=True, exist_ok=True)

WANDB_STATIC_POOLING  = 'avg'
WANDB_LOWERCASE       = False
WANDB_MIN_ITEMS       = 5

_WANDB_TABLE_KEYS = [
    'df_cat_typ_emb',
    'df_cat_typ_ctx_by_layer',
    'df_cat_typ_jac_by_layer',
    'df_cat_emb_jac_by_layer',
    'df_cat_ctx_jac_by_layer',
    'df_ll_jac',
]

_GEMMA_MODEL_MAP = {
    '270m': 'gemma-3-270m',
    '1b':   'gemma-3-1b-pt',
    '4b':   'gemma-3-4b-pt',
    '12b':  'gemma-3-12b-pt',
    '27b':  'gemma-3-27b-pt',
}


def _wcfg(cfg, *keys, default=None):
    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur


def _sae_label_from_id_template(s: str) -> Optional[str]:
    s = s.lower()
    m = re.search(r'width[_-](\d+k)[_-]l\d+[_-](big|small|medium|canonical)', s)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    m = re.search(r'(\d+k).+?(big|small|medium|canonical)', s)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    m = re.search(r'(\d+k)', s)
    if m:
        return m.group(1)
    return None


def _normalize_sae_model_name(model_part: str) -> Optional[str]:
    if model_part in MODEL_ORDER:
        return model_part
    m = re.match(r'gemma-3-?(270m|1b|4b|12b|27b)$', model_part, re.I)
    if m:
        return _GEMMA_MODEL_MAP.get(m.group(1).lower())
    if re.search(r'llama', model_part, re.I) and re.search(r'8b', model_part, re.I):
        return 'meta-llama/Llama-3.1-8B'
    if re.search(r'mistral', model_part, re.I) and re.search(r'7b', model_part, re.I):
        return 'mistralai/Mistral-7B-v0.1'
    if re.search(r'gpt2', model_part, re.I):
        return 'gpt2-small'
    return None


def _get_wandb_model_and_label(run) -> Tuple[Optional[str], Optional[str]]:
    cfg = run.config or {}
    sae = cfg.get('sae')

    # Case 1: full dict config (sae.model.name + sae.id_template)
    if isinstance(sae, dict):
        model_name  = _wcfg(sae, 'model', 'name')
        id_template = sae.get('id_template', '')
        label = _sae_label_from_id_template(id_template or '')
        if model_name and model_name in MODEL_ORDER:
            return model_name, label

    # Case 2: compact string 'model_part/width_variant'
    sae_str = str(sae) if sae is not None else ''
    if '/' in sae_str:
        model_part, label_part = sae_str.split('/', 1)
        model_name = _normalize_sae_model_name(model_part.strip())
        label      = label_part.strip().replace('_', ' ')
        if model_name:
            return model_name, label

    return None, None

# 131k -> 128k (see _width_128k); wrapped here, before the W&B runs are grouped by label below
__get_wandb_model_and_label_raw = _get_wandb_model_and_label


def _get_wandb_model_and_label(*args, **kwargs):
    model_name, label = __get_wandb_model_and_label_raw(*args, **kwargs)
    return model_name, _width_128k(label)



def _find_wandb_table(run, logical_name: str):
    pat = re.compile(rf'^{re.escape(logical_name)}_\d+_')
    hits = [f for f in run.files()
            if f.name.endswith('.table.json') and pat.match(Path(f.name).name)]
    return sorted(hits, key=lambda x: x.name)[0] if hits else None


def _parse_table_json(path: Path) -> pd.DataFrame:
    with path.open('r', encoding='utf-8') as fp:
        obj = _json.load(fp)
    return pd.DataFrame(obj['data'], columns=obj['columns'])


def _download_wandb_table(file_obj, cache_dir: Path) -> Path:
    cache_dir.mkdir(parents=True, exist_ok=True)
    return Path(file_obj.download(root=str(cache_dir), replace=False, exist_ok=True).name)


def load_tables_from_wandb(run) -> dict:
    model_name, label = _get_wandb_model_and_label(run)
    safe_model  = (model_name or 'unknown').replace('/', '__')
    safe_sae    = (label or run.name).replace('/', '__').replace(' ', '_')
    cache_dir   = WANDB_CACHE / safe_model / safe_sae

    tables = {k: None for k in _WANDB_TABLE_KEYS}
    for key in _WANDB_TABLE_KEYS:
        fobj = _find_wandb_table(run, key)
        if fobj is None:
            continue
        try:
            path = _download_wandb_table(fobj, cache_dir)
            tables[key] = _parse_table_json(path)
        except Exception as e:
            print(f'  [wandb] {key} error for {run.name}: {e}')
    return tables


# ── Build (model, release) -> label map from local data (for rename detection) ──
# Only non-standard labels (like lxr-8x, res-wg) can be renamed;
# standard labels like '262k small' are matched directly by label.
_local_nonstd: Dict[str, Dict[str, str]] = defaultdict(dict)  # model -> {release: label}
for _rdir in result_dirs:
    _rm, _rrel = parse_run_meta(_rdir, RESULTS_DIR)
    _rlbl = _read_sae_label(_rdir) or sae_row_label(_rrel)
    if _rlbl and not re.fullmatch(r'\d+k(?:\s+(?:big|small|medium|canonical))?', _rlbl):
        _local_nonstd[_rm][_rrel] = _rlbl

print(f'Fetching W&B runs from {WANDB_PROJECT} ...')
_wapi  = wandb.Api(timeout=180)
_wruns = list(_wapi.runs(f'{WANDB_ENTITY}/{WANDB_PROJECT}', per_page=200))
print(f'Found {len(_wruns)} W&B runs')

# Group by (model, label); skip runs where model or label can't be determined
_wrun_groups: Dict[Tuple[str, str], list] = defaultdict(list)
for _run in _wruns:
    _cfg = _run.config or {}
    _sp  = _cfg.get('static_pooling')
    _lc  = _cfg.get('lowercase')
    _mi  = _cfg.get('min_items_per_category')
    if _sp is not None and _sp != WANDB_STATIC_POOLING:
        continue
    if _lc is not None and bool(_lc) != WANDB_LOWERCASE:
        continue
    if _mi is not None and int(_mi) != WANDB_MIN_ITEMS:
        continue
    _model, _label = _get_wandb_model_and_label(_run)
    if _model is None or _model not in MODEL_ORDER or _label is None:
        continue
    _wrun_groups[(_model, _label)].append(_run)

# Process each (model, label) wandb group
_n_added = _n_skipped = _n_renamed = _n_empty = 0
for (_model, _label), _group in sorted(_wrun_groups.items()):
    _best = sorted(_group, key=lambda r: str(getattr(r, 'created_at', '')))[-1]

    # Check if this wandb run name matches a non-standard local release (rename case)
    _conflict_label = None
    for _crel, _clbl in _local_nonstd.get(_model, {}).items():
        if _best.name.startswith(_crel + '_'):
            _conflict_label = _clbl
            break

    if _conflict_label is not None:
        # Rename non-standard local label to match wandb label
        if _conflict_label in model_to_runs[_model] and _conflict_label != _label:
            model_to_runs[_model][_label] = model_to_runs[_model].pop(_conflict_label)
            print(f'  [wandb] renamed local {_model} / {_conflict_label} → {_label}')
            _n_renamed += 1
        else:
            print(f'  [wandb] skip (already renamed or same label): {_model} / {_label}')
            _n_skipped += 1
        continue

    # Standard label: skip if already present (local or previously downloaded)
    if _label in model_to_runs[_model]:
        print(f'  [wandb] skip (exists): {_model} / {_label}')
        _n_skipped += 1
        continue

    # Download from wandb
    print(f'  [wandb] loading {_model} / {_label} ...')
    _tables = load_tables_from_wandb(_best)
    if all(v is None for v in _tables.values()):
        print(f'  [wandb] WARNING: no table data for {_model} / {_label} — skipping')
        _n_empty += 1
        continue
    model_to_runs[_model][_label] = _tables
    _n_added += 1

print(f'W&B: added={_n_added}, skipped={_n_skipped}, renamed={_n_renamed}, no-data={_n_empty}')
print(f'Models with data: {sorted(model_to_runs.keys(), key=model_sort_key)}')
for _m in sorted(model_to_runs.keys(), key=model_sort_key):
    _labels = sorted(model_to_runs[_m].keys(), key=sae_sort_key)
    print(f'  {_m}: {_labels}')

# Rebuild ordered_cats to include categories from all sources
_all_cats: Dict[str, List[float]] = defaultdict(list)
for _runs_dict in model_to_runs.values():
    for _T in _runs_dict.values():
        _df = _T.get('df_cat_typ_jac_by_layer')
        if _df is None:
            continue
        _lc = get_layer_cols(_df)
        for _, _row in _df.iterrows():
            _all_cats[str(_row['category'])].append(
                float(np.nanmean([_row[c] for c in _lc]))
            )
ordered_cats = sorted(_all_cats.keys())[:MAX_CATEGORIES]
print(f'Updated categories: {len(ordered_cats)}, first 10: {ordered_cats[:10]}')


## Typicality vs embedding cosim (static baseline)

Average-pooled W_E cosine similarity — no template, no context.  
Shown as a line plot per model (one line per model, categories on x-axis), mirroring the original typicality vs APT_sym figure.

In [ ]:
# ── Typicality vs emb cosim: line plot (one line per model) ──────────────────
x = np.arange(len(ordered_cats))

fig_width = max(18.0, 0.34 * len(ordered_cats) + 8.0)
fig, ax   = plt.subplots(figsize=(fig_width, 9.0))

for model_name in sorted(model_to_runs.keys(), key=model_sort_key):
    # Use the first available SAE run for this model as the representative
    sae_labels = sorted(model_to_runs[model_name].keys(), key=sae_sort_key)
    T = model_to_runs[model_name][sae_labels[0]]
    if T['df_cat_typ_emb'] is None:
        continue
    y = (
        T['df_cat_typ_emb']
        .set_index('category')
        .reindex(ordered_cats)['spearman_rho']
        .to_numpy(dtype=float)
    )
    ax.plot(x, y, marker='o', linewidth=1.8, markersize=4.0,
            label=PRETTY_MODEL.get(model_name, model_name))

ax.axhline(0.0, color='black', linewidth=1.0, alpha=0.6)
ax.set_xlim(-0.5, len(ordered_cats) - 0.5)
ax.set_xticks(x)
ax.set_xticklabels(ordered_cats, rotation=45, ha='right',
                   rotation_mode='anchor', fontsize=CATEGORY_TICK_FONTSIZE)
ax.set_xlabel('Category', fontsize=LABEL_FONTSIZE)
ax.set_ylabel(r'Spearman $\rho$', fontsize=LABEL_FONTSIZE)
ax.set_title(r'$\rho_{\mathrm{S}}(s_{\mathrm{hum}},\, s_{\mathrm{emb}})$',
             fontsize=SUPTITLE_FONTSIZE, pad=16)
ax.tick_params(axis='y', labelsize=TICK_FONTSIZE)
ax.grid(axis='y', alpha=0.25)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0),
          borderaxespad=0.0, frameon=False, fontsize=LEGEND_FONTSIZE)

fig.subplots_adjust(left=0.08, right=0.79, bottom=0.30, top=0.90)
out = OUTPUT_DIR / 'experiment2_typicality_vs_apt_sym_by_model.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved: {out}')

## Main grid: per-model heatmap grids

One figure per model. Rows = SAE variants.  
Columns: `ρ(typicality, ctx cosim)` | `ρ(typicality, Jaccard)` | `ρ(ctx cosim_l, Jaccard_l)` | `ρ(Jaccard_{l1}, Jaccard_{l2})`  
Missing data shown as blank panels.

In [ ]:
# ── Per-model grid figures ────────────────────────────────────────────────────
for model_name in sorted(model_to_runs.keys(), key=model_sort_key):
    runs       = model_to_runs[model_name]
    sae_labels = sorted(runs.keys(), key=sae_sort_key)
    n_rows     = len(sae_labels)
    subplot_top = 0.80 if n_rows == 1 else MAIN_SUBPLOT_TOP
    n_cats     = len(ordered_cats)

    row_height   = max(7.5, 0.15 * n_cats)
    heat_width   = row_height
    spacer_width = 0.20 * heat_width
    # 4 rectangular panels + 1 spacer + 1 square = 6 subplot columns
    fig_width    = 4.0 * heat_width + spacer_width + heat_width + 2.3
    fig_height   = row_height * n_rows

    width_ratios = [heat_width, heat_width, heat_width, heat_width, spacer_width, heat_width]

    fig, axes = plt.subplots(
        n_rows, 6,
        figsize=(fig_width, fig_height),
        squeeze=False,
        gridspec_kw={'width_ratios': width_ratios},
    )
    fig.suptitle(PRETTY_MODEL.get(model_name, model_name),
                 fontsize=SUPTITLE_FONTSIZE, y=(0.96 if n_rows == 1 else 0.98))

    for i, sae_label in enumerate(sae_labels):
        _ref = SPECIAL_REFERENCE_TEXT.get((model_name, sae_label))
        plot_row(
            axes=axes[i],
            tables=runs[sae_label],
            row_label=sae_label,
            ordered_cats=ordered_cats,
            show_titles=(i == 0),
            last_row=(i == n_rows - 1),
            reference_text_panel1=(_ref[0] if _ref else None),
            reference_text_panel2=(_ref[1] if _ref else None),
        )

    fig.subplots_adjust(
        left=MAIN_SUBPLOT_LEFT,
        right=MAIN_SUBPLOT_RIGHT,
        bottom=MAIN_SUBPLOT_BOTTOM,
        top=subplot_top,
        hspace=0.18,
        wspace=0.08,
    )
    add_shared_colorbar(fig, bottom=MAIN_SUBPLOT_BOTTOM, top=subplot_top)

    safe = model_name.replace('/', '__').replace(' ', '_')
    out  = OUTPUT_DIR / f'{safe}__experiment2_grid.png'
    fig.savefig(out, dpi=125, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Saved: {out}')

In [ ]:
# ── Special-case standalone figures (one per entry in SPECIAL_CASES) ──────────
for model_name, sae_label in SPECIAL_CASES:
    if model_name not in model_to_runs or sae_label not in model_to_runs[model_name]:
        print(f'No data for special case {model_name} / {sae_label}, skipping.')
        continue
    tables = model_to_runs[model_name][sae_label]

    df_typ = tables['df_cat_typ_jac_by_layer']    # typicality vs Jaccard by layer
    df_ctx = tables['df_cat_ctx_jac_by_layer']    # ctx cosim vs Jaccard by layer

    if df_typ is None and df_ctx is None:
        print(f'No layer data for {model_name} / {sae_label}, skipping.')
        continue

    layer_cols   = get_layer_cols(df_typ if df_typ is not None else df_ctx)
    layer_labels = [str(int(c.split('_')[1])) for c in layer_cols]

    n_panels = (df_typ is not None) + (df_ctx is not None)
    fig_height = 7.5 * n_panels
    n_cats     = len(ordered_cats)
    panel_width = max(7.5, 0.15 * n_cats)

    fig, axes_sc = plt.subplots(n_panels, 1, figsize=(panel_width + 1.5, fig_height), squeeze=False)
    panel_idx = 0

    if df_typ is not None:
        ax = axes_sc[panel_idx, 0]
        mat = df_typ.set_index('category').reindex(ordered_cats)[layer_cols].to_numpy(dtype=float)
        plot_rect_on_ax(
            ax, mat, ordered_cats, layer_labels,
            title=COLUMN_TITLES[1],
            xlabel=r'Layer $\ell$',
            show_title=True, show_xlabel=(n_panels == 1 or panel_idx == n_panels - 1),
            show_yticks=True,
        )
        panel_idx += 1

    if df_ctx is not None:
        ax = axes_sc[panel_idx, 0]
        mat = df_ctx.set_index('category').reindex(ordered_cats)[layer_cols].to_numpy(dtype=float)
        plot_rect_on_ax(
            ax, mat, ordered_cats, layer_labels,
            title=COLUMN_TITLES[3],
            xlabel=r'Layer $\ell$',
            show_title=True, show_xlabel=True,
            show_yticks=True,
        )

    fig.subplots_adjust(left=0.20, right=0.93, bottom=0.10, top=0.97, hspace=0.20)
    add_shared_colorbar(fig, bottom=0.10, top=0.97)

    safe     = model_name.replace('/', '__').replace(' ', '_')
    sae_safe = sae_label.replace('/', '__').replace(' ', '_')
    out = OUTPUT_DIR / f'{safe}__{sae_safe}__experiment2_vertical_panels12.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Saved: {out}')

## New metrics: embedding Jaccard + clustering AMI

Load the additional CSVs produced by the updated `main.py`.
Only available for models where the new run was executed locally.

In [ ]:
# ── Load new emb-jaccard + clustering data from local result dirs ────────────
# These CSVs are produced by the updated main.py but not loaded above.

def load_new_tables(result_dir: Path) -> dict:
    def _read(name):
        p = result_dir / name
        return pd.read_csv(p) if p.exists() else None
    return {
        'df_cat_typ_emb_jac':           _read('df_cat_typ_emb_jac.csv'),
        'df_cat_emb_cosim_emb_jac':     _read('df_cat_emb_cosim_emb_jac.csv'),
        'df_clustering_ami_static':      _read('df_clustering_ami_static.csv'),
        'df_clustering_ami_contextual':  _read('df_clustering_ami_contextual.csv'),
        'df_clustering_ami_jaccard':     _read('df_clustering_ami_jaccard.csv'),
        'df_clustering_ami_emb_jaccard': _read('df_clustering_ami_emb_jaccard.csv'),
    }

model_to_new: Dict[str, Dict[str, dict]] = defaultdict(dict)
for _rdir in result_dirs:
    _model_name, _sae_release = parse_run_meta(_rdir, RESULTS_DIR)
    _row_label = _read_sae_label(_rdir) or sae_row_label(_sae_release)
    _new = load_new_tables(_rdir)
    if any(v is not None for v in _new.values()):
        model_to_new[_model_name][_row_label] = _new

print('New emb-jaccard / clustering data available for:')
for _m, _runs in sorted(model_to_new.items(), key=lambda x: model_sort_key(x[0])):
    _keys = sorted({k for t in _runs.values() for k, v in t.items() if v is not None})
    print(f'  {_m}: {sorted(_runs.keys(), key=sae_sort_key)}')
    print(f'    tables: {_keys}')

In [ ]:
# ── Pull new-metric tables from W&B (experiment_2_set_consistency_paper_clustering) ──
# Fetches: df_clustering_ami_{static,contextual,jaccard,emb_jaccard},
#          df_cat_typ_emb_jac, df_cat_emb_cosim_emb_jac

WANDB_CLUSTERING_PROJECT = WANDB_PROJECT  # main.py logs the clustering (AMI) tables to the same runs
WANDB_CLUSTERING_CACHE   = Path('.wandb_cache_clustering')
WANDB_CLUSTERING_CACHE.mkdir(parents=True, exist_ok=True)

_NEW_METRIC_TABLE_NAMES = {
    'df_cat_typ_emb_jac':           'cat_typ_emb_jac',
    'df_cat_emb_cosim_emb_jac':     'cat_emb_cosim_emb_jac',
    'df_clustering_ami_static':      'clustering_ami_static_embedding',
    'df_clustering_ami_contextual':  'clustering_ami_contextual',
    'df_clustering_ami_jaccard':     'clustering_ami_jaccard',
    'df_clustering_ami_emb_jaccard': 'clustering_ami_emb_jaccard',
}

print(f'Fetching runs from {WANDB_ENTITY}/{WANDB_CLUSTERING_PROJECT} ...')
_wruns_clust = list(_wapi.runs(f'{WANDB_ENTITY}/{WANDB_CLUSTERING_PROJECT}', per_page=200))
print(f'Found {len(_wruns_clust)} runs')

_n_wandb_new = 0
# Sort ascending by creation time so the last iteration wins (most recent run)
for _run in sorted(_wruns_clust, key=lambda r: str(getattr(r, 'created_at', ''))):
    _cfg = _run.config or {}
    _sp  = _cfg.get('static_pooling')
    _lc  = _cfg.get('lowercase')
    _mi  = _cfg.get('min_items_per_category')
    if _sp is not None and _sp != WANDB_STATIC_POOLING:
        continue
    if _lc is not None and bool(_lc) != WANDB_LOWERCASE:
        continue
    if _mi is not None and int(_mi) != WANDB_MIN_ITEMS:
        continue

    _model, _label = _get_wandb_model_and_label(_run)
    if _model is None or _model not in MODEL_ORDER or _label is None:
        continue

    _existing = dict(model_to_new.get(_model, {}).get(_label, {}))
    if all(_existing.get(k) is not None for k in _NEW_METRIC_TABLE_NAMES):
        continue  # already complete from local CSVs

    _safe_mdl  = _model.replace('/', '__')
    _safe_sae  = _label.replace('/', '__').replace(' ', '_')
    _cache_dir = WANDB_CLUSTERING_CACHE / _safe_mdl / _safe_sae

    _any_new = False
    for _key, _tname in _NEW_METRIC_TABLE_NAMES.items():
        if _existing.get(_key) is not None:
            continue
        _fobj = _find_wandb_table(_run, _tname)
        if _fobj is None:
            continue
        try:
            _path = _download_wandb_table(_fobj, _cache_dir)
            _existing[_key] = _parse_table_json(_path)
            _any_new = True
        except Exception as _e:
            print(f'  [clust] {_key} error for {_model}/{_label}: {_e}')

    if _any_new:
        if _model not in model_to_new:
            model_to_new[_model] = {}
        model_to_new[_model][_label] = _existing
        _n_wandb_new += 1
        print(f'  [clust] loaded {_model} / {_label}')

print(f'\nLoaded clustering tables from W&B for {_n_wandb_new} additional runs')
print('model_to_new after clustering pull:')
for _m in sorted(model_to_new.keys(), key=model_sort_key):
    print(f'  {PRETTY_MODEL.get(_m, _m)}: {sorted(model_to_new[_m].keys(), key=sae_sort_key)}')


In [ ]:
# ── Diagnostic: list actual W&B table files for gemma-3-270m / 16k big ───────
_diag_key = ('gemma-3-270m', '16k big')
_diag_group = _wrun_groups.get(_diag_key, [])
print(f'{_diag_key}: {len(_diag_group)} run(s) in _wrun_groups')

if _diag_group:
    _diag_run = sorted(_diag_group, key=lambda r: str(getattr(r, 'created_at', '')))[-1]
    print(f'Best run: {_diag_run.name}  (created: {getattr(_diag_run, "created_at", "?")})')
    _all_tables = [f.name for f in _diag_run.files() if '.table.json' in f.name]
    print(f'All .table.json files ({len(_all_tables)}):')
    for _fn in sorted(_all_tables):
        print(f'  {_fn}')
else:
    print('Run not found in _wrun_groups!')
    print('Keys in _wrun_groups with gemma-3-270m:')
    for k in _wrun_groups:
        if 'gemma-3-270m' in str(k):
            print(f'  {k}')


## 1×3 static-embedding comparison

Left: ρ(typicality, emb cosim) — same as above, for reference.  
Centre: ρ(typicality, emb Jaccard) — does SAE layer 0 on W_E track typicality?  
Right: ρ(emb cosim, emb Jaccard) — do static cosim and static SAE Jaccard agree?

In [ ]:
# ── 1×3 figure: ρ(typ, emb cosim) | ρ(typ, emb Jaccard) | ρ(emb cosim, emb Jaccard) ──
# Each panel is a per-category line plot (one line per model), mirroring the
# standalone typicality-vs-emb-cosim figure above.
# Panels 2 and 3 only have data for models where the new main.py was run.

TRIO_TITLES = [
    r'$\rho_{\mathrm{S}}(s_{\mathrm{hum}},\, s_{\mathrm{emb}})$',
    r'$\rho_{\mathrm{S}}(s_{\mathrm{hum}},\, s_{\mathrm{EmbJac}})$',
    r'$\rho_{\mathrm{S}}(s_{\mathrm{emb}},\, s_{\mathrm{EmbJac}})$',
]
# (source, col_key):  'orig' -> model_to_runs,  'new' -> model_to_new
TRIO_SOURCES = [
    ('orig', 'df_cat_typ_emb'),
    ('new',  'df_cat_typ_emb_jac'),
    ('new',  'df_cat_emb_cosim_emb_jac'),
]

x = np.arange(len(ordered_cats))
panel_w   = max(9.0, 0.34 * len(ordered_cats) + 4.0)
fig_width = panel_w * 3 + 1.5
fig, axes = plt.subplots(1, 3, figsize=(fig_width, 9.0), sharey=False)

for panel_idx, (ax, (src, col_key), title) in enumerate(
        zip(axes, TRIO_SOURCES, TRIO_TITLES)):
    for model_name in sorted(model_to_runs.keys(), key=model_sort_key):
        if src == 'orig':
            sae_labels = sorted(model_to_runs[model_name].keys(), key=sae_sort_key)
            df = model_to_runs[model_name][sae_labels[0]].get(col_key)
        else:
            new_runs = model_to_new.get(model_name, {})
            if not new_runs:
                continue
            new_label = sorted(new_runs.keys(), key=sae_sort_key)[0]
            df = new_runs[new_label].get(col_key)
        if df is None or 'spearman_rho' not in df.columns:
            continue
        y = df.set_index('category').reindex(ordered_cats)['spearman_rho'].to_numpy(dtype=float)
        ax.plot(x, y, marker='o', linewidth=1.8, markersize=4.0,
                label=PRETTY_MODEL.get(model_name, model_name))

    ax.axhline(0.0, color='black', linewidth=1.0, alpha=0.6)
    ax.set_xlim(-0.5, len(ordered_cats) - 0.5)
    ax.set_ylim(-1.0, 1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(ordered_cats, rotation=45, ha='right',
                       rotation_mode='anchor', fontsize=CATEGORY_TICK_FONTSIZE)
    ax.set_xlabel('Category', fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(r'Spearman $\rho$', fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=SUPTITLE_FONTSIZE, pad=16)
    ax.tick_params(axis='y', labelsize=TICK_FONTSIZE)
    ax.grid(axis='y', alpha=0.25)
    if panel_idx == 2:
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0),
                  borderaxespad=0.0, frameon=False, fontsize=LEGEND_FONTSIZE)

fig.subplots_adjust(left=0.06, right=0.82, bottom=0.30, top=0.90, wspace=0.38)
out = OUTPUT_DIR / 'experiment2_trio_emb_jaccard.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved: {out}')

## Clustering AMI vs layer

k-medoids clustering (k = #categories) on pairwise distance = 1 − similarity.  
Shows how well each representation separates the dataset categories per layer.  
Contextual and SAE Jaccard as lines; static embedding and emb Jaccard as scalars.

In [ ]:
# ── Clustering AMI vs layer — 8-row × N-col grid ─────────────────────────────
# Contextual residual + SAE Jaccard lines only (static scalars excluded).
# Shared y-scale [0, 0.5]; y-axis label on leftmost col only; uniform x ticks per row.

def _parse_layer(rep_str: str) -> Optional[int]:
    m = re.search(r'_(\d+)$', rep_str)
    return int(m.group(1)) if m else None

CTX_COLOR = '#1f77b4'
JAC_COLOR = '#ff7f0e'

# ── Build rename map: old local label → W&B canonical label ──────────────────
# Uses the already-computed _local_nonstd and _wrun_groups from cell-wandb.
# This correctly handles lxr-8x→32k, lxr-32x→131k, res-wg→65k, etc.
_rename_map: Dict[str, Dict[str, str]] = {}
for _m in MODEL_ORDER:
    _m_renames: Dict[str, str] = {}
    for (_wm, _wlabel), _wgroup in _wrun_groups.items():
        if _wm != _m:
            continue
        _wbest = sorted(_wgroup, key=lambda r: str(getattr(r, 'created_at', '')))[-1]
        for _crel, _clbl in _local_nonstd.get(_m, {}).items():
            if _wbest.name.startswith(_crel + '_'):
                _m_renames[_clbl] = _wlabel
    _rename_map[_m] = _m_renames

# ── Reconcile model_to_new labels with model_to_runs labels ──────────────────
_new_reconciled: Dict[str, Dict[str, dict]] = {}
for _m in MODEL_ORDER:
    if _m not in model_to_new:
        continue
    _renames = _rename_map.get(_m, {})
    _reconciled: Dict[str, dict] = {}
    for _k, _v in model_to_new[_m].items():
        _reconciled[_renames.get(_k, _k)] = _v
    _new_reconciled[_m] = _reconciled

# ── Grid dimensions ───────────────────────────────────────────────────────────
model_sae_labels = {
    m: sorted(model_to_runs[m].keys(), key=sae_sort_key)
    for m in MODEL_ORDER if m in model_to_runs
}
n_rows = len(MODEL_ORDER)
n_cols = max((len(v) for v in model_sae_labels.values()), default=1)
print(f'Grid: {n_rows} rows × {n_cols} cols')
for _m in MODEL_ORDER:
    if _m in model_sae_labels:
        _has = sorted(_new_reconciled.get(_m, {}).keys(), key=sae_sort_key)
        print(f'  {PRETTY_MODEL.get(_m, _m)}: {model_sae_labels[_m]}  [data: {_has}]')

Y_MAX     = 0.5
Y_TICKS   = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
N_X_TICKS = 5

fig_w = max(5.0, n_cols * 3.2) + 1.5
fig_h = n_rows * 3.2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False)

for row_idx, model_name in enumerate(MODEL_ORDER):
    pretty   = PRETTY_MODEL.get(model_name, model_name)
    new_data = _new_reconciled.get(model_name, {})

    # Row label rotated 90°; pushed further left to clear the wider left margin
    axes[row_idx, 0].annotate(
        pretty,
        xy=(0, 0.5), xycoords='axes fraction',
        xytext=(-0.42, 0.5), textcoords='axes fraction',
        ha='center', va='center', rotation=90,
        fontsize=ROWLABEL_FONTSIZE - 4, fontweight='bold',
        annotation_clip=False,
    )

    if model_name not in model_sae_labels:
        for ax in axes[row_idx]:
            ax.axis('off')
        continue

    sae_labels        = model_sae_labels[model_name]
    row_data_axes     = []   # axes where lines were actually plotted (for x-limit)
    row_active_cols   = []

    for col_idx, sae_label in enumerate(sae_labels):
        ax = axes[row_idx, col_idx]
        ax.set_title(sae_label, fontsize=TITLE_FONTSIZE - 10, pad=3)
        ax.set_ylim(0.0, Y_MAX)
        ax.set_yticks(Y_TICKS)
        ax.grid(axis='y', alpha=0.30, linewidth=0.8)

        if col_idx == 0:
            ax.set_yticklabels([f'{v:.1f}' for v in Y_TICKS],
                               fontsize=TICK_FONTSIZE - 8)
            ax.set_ylabel('AMI', fontsize=LABEL_FONTSIZE - 8)
        else:
            ax.set_yticklabels([])

        T = new_data.get(sae_label)
        if T is None:
            ax.set_xticks([])
            for sp in ax.spines.values():
                sp.set_alpha(0.3)
            ax.set_facecolor('#f8f8f8')
            continue

        row_active_cols.append(col_idx)
        has_lines = False

        df_ctx = T.get('df_clustering_ami_contextual')
        if df_ctx is not None:
            pts = sorted(
                [(r['representation'], r['ami']) for _, r in df_ctx.iterrows()
                 if _parse_layer(r['representation']) is not None],
                key=lambda x: _parse_layer(x[0]),
            )
            if pts:
                ax.plot([_parse_layer(r) for r, _ in pts], [a for _, a in pts],
                        marker='o', markersize=2.5, linewidth=1.5,
                        color=CTX_COLOR, label='Contextual')
                has_lines = True

        df_jac = T.get('df_clustering_ami_jaccard')
        if df_jac is not None:
            pts = sorted(
                [(r['representation'], r['ami']) for _, r in df_jac.iterrows()
                 if _parse_layer(r['representation']) is not None],
                key=lambda x: _parse_layer(x[0]),
            )
            if pts:
                ax.plot([_parse_layer(r) for r, _ in pts], [a for _, a in pts],
                        marker='s', markersize=2.5, linewidth=1.5,
                        color=JAC_COLOR, label='SAE Jaccard')
                has_lines = True

        ax.tick_params(labelsize=TICK_FONTSIZE - 8)
        if has_lines:
            ax.legend(fontsize=LEGEND_FONTSIZE - 10, loc='upper left', frameon=False,
                      handlelength=1.0, borderpad=0.2, labelspacing=0.2)
            row_data_axes.append(ax)

    # Blank unused columns
    for col_idx in range(len(sae_labels), n_cols):
        axes[row_idx, col_idx].axis('off')

    # Uniform 5-tick integer x-axis — tick labels on every row
    if row_data_axes:
        xmax  = max(ax.get_xlim()[1] for ax in row_data_axes)
        x_int = int(np.floor(xmax))
        ticks = np.linspace(0, x_int, N_X_TICKS, dtype=int).tolist()
        is_last_row = (row_idx == n_rows - 1)
        for col_idx in row_active_cols:
            ax = axes[row_idx, col_idx]
            ax.set_xlim(0, x_int)
            ax.set_xticks(ticks)
            ax.set_xticklabels([str(t) for t in ticks], fontsize=TICK_FONTSIZE - 8)
        if is_last_row:
            axes[row_idx, 0].set_xlabel('Layer', fontsize=LABEL_FONTSIZE - 8)

fig.subplots_adjust(left=0.15, right=0.99, top=0.98, bottom=0.04,
                    hspace=0.65, wspace=0.16)  # room between adjacent x-axes' edge tick labels

out = OUTPUT_DIR / 'experiment2_clustering_ami_by_layer.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved: {out}')


In [ ]:
# ── Static scalar table: Static Emb AMI + Static Jaccard (Emb) AMI ───────────
rows = []
for model_name in sorted(model_to_new.keys(), key=model_sort_key):
    for sae_label in sorted(model_to_new[model_name].keys(), key=sae_sort_key):
        T = model_to_new[model_name][sae_label]

        static_ami = None
        df_stat = T.get('df_clustering_ami_static')
        if df_stat is not None and len(df_stat) > 0:
            static_ami = float(df_stat.iloc[0]['ami'])

        emb_jac_ami = None
        df_ej = T.get('df_clustering_ami_emb_jaccard')
        if df_ej is not None:
            layer_rows = df_ej[df_ej['representation'].str.match(r'emb_jaccard_layer_\d+$')]
            if len(layer_rows) > 0:
                emb_jac_ami = float(layer_rows.iloc[0]['ami'])

        rows.append({
            'Model': PRETTY_MODEL.get(model_name, model_name),
            'SAE Type': sae_label,
            'Static Emb AMI': f'{static_ami:.4f}' if static_ami is not None else '—',
            'Static Jaccard (Emb) AMI': f'{emb_jac_ami:.4f}' if emb_jac_ami is not None else '—',
        })

df_scalar_table = pd.DataFrame(rows)
print(df_scalar_table.to_string(index=False))


In [ ]:
# ── 2×4 per-model AMI overview: embedding + all Jaccard SAE types ─────────────
# Layout: 4 rows × 2 cols (one panel per model, MODEL_ORDER left→right top→bottom).
# Each panel:
#   black thick line  = embedding AMI (SAE-independent; one representative SAE chosen)
#   coloured lines    = SAE Jaccard AMI, one line per SAE type
# Special case: gemma-3-270m / 16k big is shown in a separate standalone figure
#               and therefore excluded from the 270m panel here.
# Shared y-scale [0, 0.5].  Legend outside the top-right edge.

_SAE_STYLES_OVERVIEW = {
    '16k big':    ('royalblue',      '-',  'o'),
    '16k medium': ('dodgerblue',     '--', 's'),
    '16k small':  ('deepskyblue',    ':',  '^'),
    '262k big':   ('crimson',        '-',  'o'),
    '262k medium':('magenta',        '--', 's'),
    '262k small': ('darkorange',     ':',  '^'),
    '32k':        ('forestgreen',    '-',  'o'),
    '128k':       ('mediumseagreen', '--', 's'),
    '131k':       ('mediumpurple',   '-',  'o'),
    '65k':        ('sienna',         '-',  'o'),
}
_FALLBACK_COLORS = ['#e377c2', '#bcbd22', '#17becf']

# Special-case SAE types excluded from their model's panel in this figure
_PANEL_EXCLUDE = {
    'gemma-3-270m': {'16k big'},
}

def _pick_ctx_sae(model_name: str, available: list) -> Optional[str]:
    is_gemma = 'gemma' in model_name.lower()
    if is_gemma:
        prefs = ['16k big', '262k big', '16k small', '262k small',
                 '262k medium', '16k medium']
    else:
        prefs = sorted(available, key=sae_sort_key)
    for p in prefs:
        if p in available:
            return p
    return available[0] if available else None

N_PANEL_ROWS, N_PANEL_COLS = 4, 2
PANEL_W, PANEL_H = 7.5, 3.8

fig3, axes3 = plt.subplots(
    N_PANEL_ROWS, N_PANEL_COLS,
    figsize=(N_PANEL_COLS * PANEL_W + 3.5, N_PANEL_ROWS * PANEL_H),
    squeeze=False,
)

legend_handles: list = []
legend_labels:  list = []
_seen_leg:      set  = set()

def _add_legend_once(h, lbl):
    if lbl not in _seen_leg:
        legend_handles.append(h); legend_labels.append(lbl); _seen_leg.add(lbl)

for panel_idx, model_name in enumerate(MODEL_ORDER):
    row = panel_idx // N_PANEL_COLS
    col = panel_idx % N_PANEL_COLS
    ax  = axes3[row, col]

    pretty   = PRETTY_MODEL.get(model_name, model_name)
    new_data = _new_reconciled.get(model_name, {})
    excluded = _PANEL_EXCLUDE.get(model_name, set())

    ax.set_title(pretty, fontsize=TITLE_FONTSIZE - 8, pad=5)
    ax.set_ylim(0.0, Y_MAX)
    ax.set_yticks(Y_TICKS)
    ax.grid(axis='y', alpha=0.30, linewidth=0.8)
    ax.set_ylabel('AMI', fontsize=LABEL_FONTSIZE - 10)

    available_saes = sorted(new_data.keys(), key=sae_sort_key)
    if not available_saes:
        ax.text(0.5, 0.5, 'no data', transform=ax.transAxes,
                ha='center', va='center', color='gray', fontsize=14)
        continue

    # ── Embedding line (black, thick; SAE-independent) ────────────────────────
    ctx_sae = _pick_ctx_sae(model_name, available_saes)
    if ctx_sae is not None:
        df_ctx = new_data[ctx_sae].get('df_clustering_ami_contextual')
        if df_ctx is not None:
            pts = sorted(
                [(r['representation'], r['ami']) for _, r in df_ctx.iterrows()
                 if _parse_layer(r['representation']) is not None],
                key=lambda x: _parse_layer(x[0]),
            )
            if pts:
                h, = ax.plot(
                    [_parse_layer(r) for r, _ in pts], [a for _, a in pts],
                    color='black', linestyle='-', linewidth=2.2, marker=None,
                    label='Embedding',
                )
                _add_legend_once(h, 'Embedding')

    # ── SAE Jaccard lines (skip excluded types) ───────────────────────────────
    _fallback_idx = 0
    for sae_label in available_saes:
        if sae_label in excluded:
            continue
        df_jac = new_data[sae_label].get('df_clustering_ami_jaccard')
        if df_jac is None:
            continue
        pts = sorted(
            [(r['representation'], r['ami']) for _, r in df_jac.iterrows()
             if _parse_layer(r['representation']) is not None],
            key=lambda x: _parse_layer(x[0]),
        )
        if not pts:
            continue
        if sae_label in _SAE_STYLES_OVERVIEW:
            color, ls, mk = _SAE_STYLES_OVERVIEW[sae_label]
        else:
            color = _FALLBACK_COLORS[_fallback_idx % len(_FALLBACK_COLORS)]
            ls, mk = '-', 'x'
            _fallback_idx += 1
        h, = ax.plot(
            [_parse_layer(r) for r, _ in pts], [a for _, a in pts],
            color=color, linestyle=ls, linewidth=1.4, marker=mk,
            markersize=3.5, label=sae_label,
        )
        _add_legend_once(h, sae_label)

    ax.tick_params(labelsize=TICK_FONTSIZE - 8)
    x_int = int(np.floor(ax.get_xlim()[1]))
    if x_int > 0:
        ticks = np.linspace(0, x_int, N_X_TICKS, dtype=int).tolist()
        ax.set_xticks(ticks)
        ax.set_xticklabels([str(t) for t in ticks], fontsize=TICK_FONTSIZE - 8)
    if row == N_PANEL_ROWS - 1:
        ax.set_xlabel('Layer', fontsize=LABEL_FONTSIZE - 10)

fig3.subplots_adjust(left=0.07, right=0.74, top=0.97, bottom=0.07,
                     hspace=0.60, wspace=0.25)

fig3.legend(
    legend_handles, legend_labels,
    loc='upper left', bbox_to_anchor=(0.76, 0.97),
    fontsize=LEGEND_FONTSIZE - 4, frameon=True,
    handlelength=2.2, labelspacing=0.55, borderpad=0.9,
    title='SAE type', title_fontsize=LEGEND_FONTSIZE - 2,
)

out3 = OUTPUT_DIR / 'experiment2_clustering_ami_per_model_overview.png'
fig3.savefig(out3, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig3)
print(f'Saved: {out3}')


In [ ]:
# ── Max AMI table: maxima of curves in experiment2_clustering_ami_per_model_overview ──
# Embedding column = max contextual residual AMI across all layers (black line).
# SAE Jaccard columns = max SAE Jaccard AMI across all layers (coloured lines).
# Uses _new_reconciled, same data source as the overview figure above.

def _max_ami(df, rep_pattern):
    if df is None or len(df) == 0:
        return None
    rows = df[df['representation'].str.match(rep_pattern)]
    return float(rows['ami'].max()) if len(rows) > 0 else None

# Collect all SAE types across all models (for column ordering)
_all_sae = sorted(
    {lbl for m in _new_reconciled for lbl in _new_reconciled[m]},
    key=sae_sort_key,
)

_rows = []
for _m in MODEL_ORDER:
    if _m not in _new_reconciled:
        continue
    _nd = _new_reconciled[_m]

    # Embedding: use max across all SAE dirs for this model (same model = same residual stream)
    _emb_max = None
    for _lbl, _T in _nd.items():
        _v = _max_ami(_T.get('df_clustering_ami_contextual'), r'contextual_layer_\d+$')
        if _v is not None:
            _emb_max = max(_emb_max, _v) if _emb_max is not None else _v

    # SAE Jaccard: per SAE type
    _jac = {}
    for _lbl, _T in _nd.items():
        _v = _max_ami(_T.get('df_clustering_ami_jaccard'), r'jaccard_layer_\d+$')
        if _v is not None:
            _jac[_lbl] = _v

    _rows.append((_m, _emb_max, _jac))

# ── Print table ───────────────────────────────────────────────────────────────
_W0 = 14  # model
_W1 = 9   # embedding
_WS = 9   # each SAE col

_hdr_saes = _all_sae
_col_w    = [_W0, _W1] + [max(_WS, len(s) + 2) for s in _hdr_saes]

def _sep(c='-'):
    return '+' + '+'.join(c * w for w in _col_w) + '+'

def _row(*cells):
    return '|' + '|'.join(str(v).center(w) for v, w in zip(cells, _col_w)) + '|'

def _f(v):
    return f'{v:.3f}' if v is not None else '—'

print()
print('  Max AMI — peaks of curves in experiment2_clustering_ami_per_model_overview')
print('  Embedding = max contextual residual AMI across layers (black line)')
print('  SAE cols  = max SAE Jaccard AMI across layers (coloured lines)')
print()
print(_sep('='))
print(_row('Model', 'Embedding', *_hdr_saes))
print(_sep('='))
for _m, _emb, _jac in _rows:
    _sae_vals = [_jac.get(s) for s in _hdr_saes]
    print(_row(PRETTY_MODEL.get(_m, _m), _f(_emb), *[_f(v) for v in _sae_vals]))
    print(_sep('-'))
print(_sep('='))


In [ ]:
# ── Special case: Gemma 3 270M — Embedding + 16k big Jaccard ─────────────────
# Standalone single-panel figure mirroring the role of cell-8 for the heatmaps.

_SC_MODEL = 'gemma-3-270m'
_SC_SAE   = '16k big'
_sc_data  = _new_reconciled.get(_SC_MODEL, {}).get(_SC_SAE)

if _sc_data is None:
    print(f'No clustering data for {_SC_MODEL} / {_SC_SAE} — skipping special-case figure.')
else:
    fig_sc, ax_sc = plt.subplots(figsize=(7.0, 3.8))

    ax_sc.set_title(
        f'{PRETTY_MODEL.get(_SC_MODEL, _SC_MODEL)} — {_SC_SAE}',
        fontsize=TITLE_FONTSIZE - 8, pad=5,
    )
    ax_sc.set_ylim(0.0, Y_MAX)
    ax_sc.set_yticks(Y_TICKS)
    ax_sc.set_yticklabels([f'{v:.1f}' for v in Y_TICKS], fontsize=TICK_FONTSIZE - 8)
    ax_sc.set_ylabel('AMI', fontsize=LABEL_FONTSIZE - 10)
    ax_sc.grid(axis='y', alpha=0.30, linewidth=0.8)

    # Embedding line (contextual, SAE-independent)
    df_ctx_sc = _sc_data.get('df_clustering_ami_contextual')
    if df_ctx_sc is not None:
        pts = sorted(
            [(r['representation'], r['ami']) for _, r in df_ctx_sc.iterrows()
             if _parse_layer(r['representation']) is not None],
            key=lambda x: _parse_layer(x[0]),
        )
        if pts:
            ax_sc.plot(
                [_parse_layer(r) for r, _ in pts], [a for _, a in pts],
                color='black', linestyle='-', linewidth=2.2, marker=None,
                label='Embedding',
            )

    # 16k big Jaccard line
    df_jac_sc = _sc_data.get('df_clustering_ami_jaccard')
    if df_jac_sc is not None:
        pts = sorted(
            [(r['representation'], r['ami']) for _, r in df_jac_sc.iterrows()
             if _parse_layer(r['representation']) is not None],
            key=lambda x: _parse_layer(x[0]),
        )
        if pts:
            color_sc, ls_sc, mk_sc = _SAE_STYLES_OVERVIEW[_SC_SAE]
            ax_sc.plot(
                [_parse_layer(r) for r, _ in pts], [a for _, a in pts],
                color=color_sc, linestyle=ls_sc, linewidth=1.4,
                marker=mk_sc, markersize=3.5, label=_SC_SAE,
            )

    ax_sc.tick_params(labelsize=TICK_FONTSIZE - 8)
    x_int_sc = int(np.floor(ax_sc.get_xlim()[1]))
    if x_int_sc > 0:
        ticks_sc = np.linspace(0, x_int_sc, N_X_TICKS, dtype=int).tolist()
        ax_sc.set_xticks(ticks_sc)
        ax_sc.set_xticklabels([str(t) for t in ticks_sc], fontsize=TICK_FONTSIZE - 8)
    ax_sc.set_xlabel('Layer', fontsize=LABEL_FONTSIZE - 10)

    ax_sc.legend(fontsize=LEGEND_FONTSIZE - 6, loc='upper left', frameon=False,
                 handlelength=2.0, borderpad=0.3, labelspacing=0.3)

    fig_sc.tight_layout()
    out_sc = OUTPUT_DIR / 'experiment2_clustering_ami_gemma270m_16kbig.png'
    fig_sc.savefig(out_sc, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig_sc)
    print(f'Saved: {out_sc}')


In [ ]:
# ── Combined Figure 2: AMI (a) + colour grids (b, c) — Gemma 3 270M / 16k big ────
_SC_MODEL_F2 = 'gemma-3-270m'
_SC_SAE_F2   = '16k big'

_sc_data_f2   = _new_reconciled.get(_SC_MODEL_F2, {}).get(_SC_SAE_F2)
_sc_tables_f2 = model_to_runs.get(_SC_MODEL_F2, {}).get(_SC_SAE_F2)

if _sc_tables_f2 is None:
    print(f'Missing heatmap data for combined Figure 2 ({_SC_MODEL_F2} / {_SC_SAE_F2}) — skipping.')
else:
    _df_typ_f2 = _sc_tables_f2['df_cat_typ_jac_by_layer']
    _df_ctx_f2 = _sc_tables_f2['df_cat_ctx_jac_by_layer']

    _n_cats_f2 = len(ordered_cats)
    _heat_h_f2 = max(7.5, 0.15 * _n_cats_f2)
    _heat_w_f2 = max(7.5, 0.15 * _n_cats_f2)
    _ami_w_f2  = 11.0  # broader aspect ratio
    _sp_w_f2   = 4.5   # wide spacer to clear long y-axis category labels of panel (b)

    # 4 columns: AMI | spacer | heatmap_b | heatmap_c
    fig_f2, _axes_f2 = plt.subplots(
        1, 4,
        figsize=(_ami_w_f2 + _sp_w_f2 + 2 * _heat_w_f2 + 3.0, _heat_h_f2),
        gridspec_kw={'width_ratios': [_ami_w_f2, _sp_w_f2, _heat_w_f2, _heat_w_f2]},
        squeeze=False,
    )
    _ax_ami, _ax_sp, _ax_b, _ax_c = _axes_f2[0]
    _ax_sp.axis('off')

    # ── Panel a): AMI vs layer ───────────────────────────────────────────────
    if _sc_data_f2 is None:
        _ax_ami.set_xticks([])
        _ax_ami.set_yticks([])
        for _sp in _ax_ami.spines.values():
            _sp.set_visible(True); _sp.set_alpha(0.25)
        _ax_ami.set_facecolor('#f8f8f8')
        _ax_ami.text(0.5, 0.5, 'AMI data not available\nfor 16k big',
                     transform=_ax_ami.transAxes, ha='center', va='center',
                     fontsize=LABEL_FONTSIZE - 4, style='italic', color='#888888',
                     multialignment='center')
        print('Note: AMI data for gemma-3-270m / 16k big not found — panel a) shows placeholder.')
    else:
        _ax_ami.set_ylim(0.0, Y_MAX)
        _ax_ami.set_yticks(Y_TICKS)
        _ax_ami.set_yticklabels([f'{v:.1f}' for v in Y_TICKS], fontsize=TICK_FONTSIZE - 4)
        _ax_ami.set_ylabel('AMI', fontsize=LABEL_FONTSIZE)
        _ax_ami.grid(axis='y', alpha=0.30, linewidth=0.8)

        _dfc_ami = _sc_data_f2.get('df_clustering_ami_contextual')
        if _dfc_ami is not None:
            _pts_a = sorted(
                [(r['representation'], r['ami']) for _, r in _dfc_ami.iterrows()
                 if _parse_layer(r['representation']) is not None],
                key=lambda x: _parse_layer(x[0]),
            )
            if _pts_a:
                _ax_ami.plot(
                    [_parse_layer(r) for r, _ in _pts_a], [a for _, a in _pts_a],
                    color='black', linestyle='-', linewidth=2.2, marker=None,
                    label=r'Embedding $s_{\mathrm{cos}}^\ell$',
                )

        _dfj_ami = _sc_data_f2.get('df_clustering_ami_jaccard')
        if _dfj_ami is not None:
            _pts_j = sorted(
                [(r['representation'], r['ami']) for _, r in _dfj_ami.iterrows()
                 if _parse_layer(r['representation']) is not None],
                key=lambda x: _parse_layer(x[0]),
            )
            if _pts_j:
                _col_f2, _ls_f2, _mk_f2 = _SAE_STYLES_OVERVIEW[_SC_SAE_F2]
                _ax_ami.plot(
                    [_parse_layer(r) for r, _ in _pts_j], [a for _, a in _pts_j],
                    color='magenta', linestyle=_ls_f2, linewidth=2.5,
                    marker=_mk_f2, markersize=5.5, label=r'SAE $s_{\mathrm{Jac}}^\ell$',
                )

        _ax_ami.tick_params(labelsize=TICK_FONTSIZE - 4)
        _xlim_f2 = int(np.floor(_ax_ami.get_xlim()[1]))
        if _xlim_f2 > 0:
            _xticks_f2 = np.linspace(0, _xlim_f2, N_X_TICKS, dtype=int).tolist()
            _ax_ami.set_xticks(_xticks_f2)
            _ax_ami.set_xticklabels([str(t) for t in _xticks_f2], fontsize=TICK_FONTSIZE - 4)
        _ax_ami.set_xlabel(r'Layer $\ell$', fontsize=LABEL_FONTSIZE)
        _ax_ami.legend(fontsize=LEGEND_FONTSIZE, loc='upper left', frameon=False,
                       handlelength=2.0, borderpad=0.3, labelspacing=0.3)

    # ── Panel b): rho(s_hum, s_jac) heatmap ─────────────────────────────────
    if _df_typ_f2 is not None:
        _lc_b  = get_layer_cols(_df_typ_f2)
        _ll_b  = [str(int(c.split('_')[1])) for c in _lc_b]
        _mat_b = _df_typ_f2.set_index('category').reindex(ordered_cats)[_lc_b].to_numpy(dtype=float)
        plot_rect_on_ax(
            _ax_b, _mat_b, ordered_cats, _ll_b,
            title=COLUMN_TITLES[1],
            xlabel=r'Layer $\ell$',
            show_title=True, show_xlabel=True, show_yticks=True,
        )
    else:
        draw_blank(_ax_b)
        _ax_b.set_title(COLUMN_TITLES[1], fontsize=TITLE_FONTSIZE, pad=18)

    # ── Panel c): rho(s_cos, s_jac) heatmap ─────────────────────────────────
    if _df_ctx_f2 is not None:
        _lc_c  = get_layer_cols(_df_ctx_f2)
        _ll_c  = [str(int(c.split('_')[1])) for c in _lc_c]
        _mat_c = _df_ctx_f2.set_index('category').reindex(ordered_cats)[_lc_c].to_numpy(dtype=float)
        plot_rect_on_ax(
            _ax_c, _mat_c, ordered_cats, _ll_c,
            title=COLUMN_TITLES[3],
            xlabel=r'Layer $\ell$',
            show_title=True, show_xlabel=True, show_yticks=False,
        )
    else:
        draw_blank(_ax_c)
        _ax_c.set_title(COLUMN_TITLES[3], fontsize=TITLE_FONTSIZE, pad=18)

    # ── Panel labels a) b) c) — above top-left of each panel ────────────────
    for _ax_lbl, _lbl in zip([_ax_ami, _ax_b, _ax_c], ['a', 'b', 'c']):
        _ax_lbl.annotate(
            f'{_lbl})',
            xy=(0, 1), xycoords='axes fraction',
            xytext=(0, 10), textcoords='offset points',
            ha='left', va='bottom',
            fontsize=LABEL_FONTSIZE, fontweight='bold',
            annotation_clip=False,
        )

    fig_f2.subplots_adjust(left=0.06, right=0.935, bottom=0.08, top=0.93, wspace=0.05)
    add_shared_colorbar(fig_f2, bottom=0.08, top=0.93, x=0.945)

    _out_f2 = OUTPUT_DIR / 'experiment2_combined_ami_grids_gemma270m_16kbig.png'
    fig_f2.savefig(_out_f2, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig_f2)
    print(f'Saved: {_out_f2}')
